# End-to-end data products: database to device

Build and reason about a production-shaped scientific data product—from validated input and durable
storage through Python services and an HTTP API (a documented request/response interface between
programs) to a responsive browser client, deployment platform, and observable operation.

**Lecture 8 · Notebook 00 · CMOR 438 / INDE 577**

## Orientation: “end to end” names a responsibility boundary

**Live core:** one measurement travels through validation, SQLite, a repository, domain service,
FastAPI endpoint, serialized JSON, and responsive HTML/JavaScript client.

**Practice:** trace failures, review API contracts, design a deployment, create a test matrix, and
separate component tests from genuinely deployed end-to-end tests.

**Extension:** hosting, containers, AWS mappings, schema migrations, CI/CD, observability, security,
ML serving, queues, browser/device emulation, operating-system matrices, and incident response.

Nothing launches a server, opens a browser, or contacts a network. FastAPI's in-process test client
exercises the package-backed vertical slice deterministically.

## How to use this notebook

**Estimated time:** 225 minutes core, plus 180 minutes of practice and extension. This is a reference;
the instructor will select a three-hour live path.

Run `uv sync`, run the course setup script, open this notebook in VS Code, select the **Rice DSM**
kernel, restart, and run all. Before each request, predict the database rows, service decision, HTTP
status, JSON body, frontend state, log/metric evidence, and failure visible to the user.

The executable example is deliberately small. Its layers are not a recommendation to create a
microservice for every function; they expose seams where real teams assign contracts and tests.

## Learning objectives

By the end, you should be able to:

- distinguish a data pipeline, request path, product, deployment pipeline, and end-to-end test;
- define the exact start and end of an “end-to-end” claim;
- trace one value across database, repository, service, API, network contract, and frontend;
- use typed validation, idempotency keys, bounded queries, API versioning, and explicit errors;
- distinguish liveness from readiness and propagate a safe correlation identifier;
- explain browser responsibilities: semantic HTML, state, accessibility, responsive layout, and safe
  rendering;
- distinguish responsive testing, browser emulation, OS simulation, device emulation, and real-device
  validation;
- map local components to hosting roles such as DNS, TLS/CDN, load balancer/API gateway, compute,
  managed database, object store, secrets, and telemetry;
- explain immutable artifacts, configuration, migrations, rollout, rollback, CI, delivery, and
  deployment;
- choose unit, contract, integration, component, smoke, end-to-end, and production synthetic tests;
- reason about authentication, authorization, CORS, CSP, injection, rate limits, privacy, and supply
  chain risk; and
- extend the path to batch features, model serving, queues, monitoring, and retraining without hiding
  scientific validity assumptions.

## Why this matters

A model or analysis creates value only when a person or another system can use it reliably. Real
failures happen between layers: a timestamp loses its timezone, a retry creates two observations, a
schema changes before the frontend, a health check lies, a mobile layout hides the error state, or a
deployment updates code before its database migration is compatible.

“The notebook worked” establishes only one local execution. Professional data scientists understand
enough of the surrounding system to make data meaning, model behavior, software contracts, and
operational evidence agree.

## Worked example: a scientific measurement monitor

An instrument submits temperature observations. The backend validates units and time, stores each
instrument event once, assigns an illustrative risk level, and exposes recent measurements. A small
web interface fetches and renders the result on narrow and wide screens.

The thresholds are teaching policy, not a scientifically validated safety model. That distinction is
part of the product contract.

```text
instrument or phone
  │ JSON over HTTPS: idempotency key, sensor, time, temperature
  ▼
edge / DNS / TLS / rate limit
  ▼
FastAPI transport adapter ── request validation and HTTP semantics
  ▼
MeasurementService ── domain policy
  ▼
SQLiteMeasurementRepository ── parameterized transaction
  ▼
database

browser ← HTML/CSS/JS ← GET JSON ← same service/repository/database
             │
             └── loading / empty / success / stale / error / offline states
```

Locally these run in one Python process. In production, each arrow can cross a process, network,
identity, ownership, version, scaling, and failure boundary.

## Professional practice: ask for the contract at every arrow

| Boundary | Questions |
| --- | --- |
| producer → API | schema, units, identity, ordering, retry, authentication |
| API → service | validated types, domain errors, authorization context |
| service → repository | transaction, idempotency, query bounds, failure semantics |
| backend → frontend | HTTP status, JSON schema, version, cache, latency, accessibility |
| build → runtime | immutable artifact, configuration, secret source, provenance |
| release → database | compatible migration order, backup, rollback limitation |
| runtime → operators | logs, metrics, traces, SLOs, alerts, runbook ownership |

Architecture is the set of contracts and tradeoffs, not merely a box diagram.

## System glossary: name each piece precisely

API stands for **application programming interface**. It is a contract that lets one software
component use another without knowing all of its implementation. Python functions and packages have
in-process APIs. This notebook focuses on an **HTTP web API**: a client sends a request to a server,
and the server returns a response. The API is the agreement about operations, input, output, errors,
authorization, limits, and compatibility—not the database, not JSON alone, and not the server machine.

For `GET /api/v1/measurements/latest?limit=20`, `GET` is the HTTP method, the `/api/.../latest` part is
the path, `limit=20` is a query parameter, and method plus path identify an endpoint. Headers carry
metadata; an optional body carries input; the response contains a status code, headers, and optional
result or error data. See [What is an API?](../../supplementary-materials/computing-foundations/07-what-is-an-api.md)
for a slower introduction and diagrams.

| Term | Working definition |
| --- | --- |
| database | organized persisted data plus its logical structure |
| DBMS | software that stores, queries, coordinates, and protects database state |
| driver | language-specific implementation of a database protocol |
| repository | application adapter hiding storage queries behind domain-oriented operations |
| migration | versioned change to persistent schema/data needed by software releases |
| backend | server-side code implementing API, policy, orchestration, and integrations |
| domain service | code expressing business/scientific policy independent of HTTP and storage details |
| API | application programming interface: supported contract through which software components interact |
| HTTP method | requested operation category, such as `GET` for retrieval or `POST` for submission |
| path | URL portion naming a web resource or operation |
| endpoint | one web-API operation identified by a method and path |
| request | method, path, headers, and optional query parameters/body sent by a client |
| response | status code, headers, and optional result/error body returned by a server |
| ASGI server | process, such as Uvicorn, that speaks network protocols and invokes a Python ASGI app |
| frontend | code/assets that present state and interaction to a user, commonly in a browser |
| client | browser, mobile app, instrument, script, or service that calls an API |
| DNS | maps a service name to routing targets; it does not encrypt traffic |
| TLS/HTTPS | authenticates a server endpoint and encrypts HTTP in transit |
| reverse proxy/load balancer | accepts traffic and routes it to healthy application instances |
| API gateway | managed API edge that may add authentication, quotas, routing, and transformations |
| CDN | geographically distributed cache/delivery layer for suitable content |
| WAF | rule-based web request filtering; one control, not complete application security |

### Runtime, delivery, and operations glossary

| Term | Working definition |
| --- | --- |
| process | one running program with memory, resources, and an operating-system identity |
| image | immutable packaged filesystem/config used to create containers |
| container | isolated process created from an image and sharing a host kernel |
| VM | virtual machine with a guest operating system and virtualized hardware |
| registry | service storing versioned artifacts such as container images or models |
| orchestrator | platform that schedules, replaces, scales, and networks workloads |
| queue/stream | asynchronous buffer/log separating producers from consumers |
| cache | derived copy used to reduce latency/load under explicit freshness rules |
| secret manager | controlled system for storing, authorizing, rotating, and auditing secrets |
| CI | frequent integration supported by automated build/test evidence |
| continuous delivery | keeping a tested immutable artifact ready for controlled release |
| continuous deployment | automatically releasing every change that satisfies policy |
| log | discrete event record |
| metric | numerical time-series aggregation |
| trace/span | causal timing path across operations and services |
| RUM | real-user monitoring collected from actual client sessions |
| SLI/SLO | measured reliability indicator and its target |

These are role definitions. One managed product may implement several roles; one role may also be split
across multiple products.

### One system, five complementary diagrams

We will use different diagrams because each answers a different question:

1. **Context:** people and external systems around the product.
2. **Container/component:** deployable/runtime pieces and dependencies.
3. **Sequence:** time-ordered behavior for one request.
4. **Deployment:** where artifacts run and how traffic/state move.
5. **CI/CD:** how a change becomes a monitored release.

A diagram without a legend, boundary, direction, or named responsibility is decoration. Keep diagrams
in version control and update them when contracts change.

## 1. Several different pipelines coexist

- A **data pipeline** ingests, validates, transforms, and publishes data.
- A **request path** handles one user/system interaction and returns a response.
- A **training pipeline** creates features, trains, evaluates, and registers a candidate model.
- A **serving pipeline** produces predictions online or in a batch.
- A **deployment pipeline** tests and promotes a software/model artifact through environments.
- A **product** includes software, data, interface, operations, documentation, policy, and people.

Calling all six “the pipeline” makes failure ownership and evidence ambiguous.

### What does end-to-end mean?

It means “from an explicitly named beginning to an explicitly named end, through the real boundaries
claimed by the test or workflow.” Examples:

- instrument event → durable curated row;
- browser action → visible confirmation;
- Git commit → running production artifact;
- raw snapshot → registered model → served prediction.

An in-process API test with a temporary SQLite database is a valuable **component/vertical-slice
test**. It is not a deployed end-to-end test through DNS, TLS, a production database engine, cloud IAM,
and a real browser. Name the boundary honestly.

### End to end does not mean test everything in one enormous test

One slow test cannot localize every failure, cover every input, or prove scientific correctness.
Teams combine fast narrow tests with fewer broad tests. A useful end-to-end test follows one critical
user journey while unit, property, contract, and integration tests cover the combinatorial detail.

### System context diagram

```mermaid
flowchart LR
    Scientist[Scientist / operator] -->|views measurements| Product[Measurement data product]
    Instrument[Laboratory instrument] -->|submits observation| Product
    Product -->|stores/query state| Database[(Managed database)]
    Product -->|archives files| ObjectStore[(Object storage)]
    Product -->|events and signals| Telemetry[Monitoring platform]
    OnCall[On-call engineer] -->|investigates alerts| Telemetry
    Identity[Identity provider] -->|identity and claims| Product
```

The product boundary includes frontend and backend responsibility; database, storage, monitoring, and
identity are dependencies with separate owners and failure modes.

### Component and request-sequence diagrams

```mermaid
flowchart TD
    Browser[Responsive browser client] --> API[FastAPI transport adapter]
    Instrument[Instrument client] --> API
    API --> Service[MeasurementService]
    Service --> Repo[Repository]
    Repo --> DB[(SQLite locally / PostgreSQL remotely)]
    API --> Assets[Packaged HTML, CSS, JavaScript]
```

```mermaid
sequenceDiagram
    participant I as Instrument
    participant A as API
    participant S as Service
    participant R as Repository
    participant D as Database
    I->>A: POST measurement + idempotency key
    A->>A: authenticate, authorize, validate
    A->>S: typed MeasurementInput
    S->>R: save(command)
    R->>D: parameterized transaction
    D-->>R: canonical row
    R-->>S: row + created flag
    S-->>A: MeasurementOutput
    A-->>I: 201 created or 200 replay + request ID
```

## 2. Confirm the executable environment

In [ ]:
import hashlib
import inspect
import json
import platform
import sys
import tomllib
from datetime import UTC, datetime
from pathlib import Path
from tempfile import TemporaryDirectory

import fastapi
import httpx2
import pydantic
import uvicorn
from fastapi.testclient import TestClient
from pydantic import ValidationError

from rice_dsm.data_product import (
    MeasurementInput,
    MeasurementService,
    SQLiteMeasurementRepository,
    create_app,
    dashboard_html,
)

print("Python:  ", sys.version.split()[0])
print("OS:      ", platform.system(), platform.machine())
print("FastAPI: ", fastapi.__version__)
print("Pydantic:", pydantic.__version__)
print("HTTPX2:  ", httpx2.__version__)
print("Uvicorn: ", uvicorn.__version__)

assert sys.version_info >= (3, 12)
assert int(fastapi.__version__.split(".")[0]) == 0
assert int(pydantic.__version__.split(".")[0]) >= 2

FastAPI defines the web application and OpenAPI contract. Pydantic validates typed request/response
models. Uvicorn is an ASGI server used when a process actually listens on a socket. HTTPX2 supports
the installed Starlette/FastAPI test client. A library, framework, server process, and hosting platform
are different layers.

## 3. Create an isolated application instance

In [ ]:
def find_project_root(start: Path) -> Path:
    """Find the nearest parent containing the course project definition."""
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("could not locate pyproject.toml")


project_root = find_project_root(Path.cwd())
application_workspace = TemporaryDirectory()
application_database_path = Path(application_workspace.name) / "measurements.sqlite3"
application = create_app(application_database_path)
client = TestClient(application)

assert application_database_path.is_file()
assert application.title == "Rice DSM Measurement API"

`create_app` is an application factory: tests and deployments can construct an app with explicit
configuration rather than importing a hidden global database. The temporary path is cross-platform.
No port is opened; requests will run against the ASGI application in process.

## 4. Organize dependencies toward the domain

```text
HTTP routes / HTML adapter
          ↓
MeasurementService
          ↓
repository interface/implementation
          ↓
SQLite driver and database
```

Transport details should not define domain policy, and SQL rows should not leak directly into public
JSON. This direction makes policy testable without HTTP and makes transport changes less invasive.
It does not require a class for every three lines; add a seam where behavior, ownership, or change
rate genuinely differs.

In [ ]:
import rice_dsm.data_product as data_product

public_definitions = {
    name
    for name, value in inspect.getmembers(data_product)
    if inspect.isclass(value) or inspect.isfunction(value)
}

assert {
    "MeasurementInput",
    "MeasurementService",
    "SQLiteMeasurementRepository",
    "create_app",
    "dashboard_html",
} <= public_definitions

## 5. Validate at the public boundary

The request model requires a stable idempotency key, sensor identity, timezone-aware timestamp, and
temperature at or above absolute zero. It rejects unknown fields. Validation protects the contract;
it does not authenticate the producer or prove the measurement is physically accurate.

In [ ]:
valid_command = MeasurementInput(
    idempotency_key="instrument-run-0042",
    sensor_id="sensor-alpha",
    observed_at=datetime(2026, 8, 29, 14, 30, tzinfo=UTC),
    temperature_c=84.5,
)

assert valid_command.sensor_id == "sensor-alpha"
assert valid_command.observed_at.utcoffset() is not None
assert valid_command.model_dump(mode="json")["observed_at"].endswith("Z")

### Invalid input becomes a structured, testable failure

In [ ]:
try:
    MeasurementInput(
        idempotency_key="instrument-run-0043",
        sensor_id="sensor-alpha",
        observed_at=datetime(2026, 8, 29, 14, 30),
        temperature_c=-300.0,
    )
except ValidationError as error:
    validation_errors = error.errors()
else:
    raise AssertionError("invalid input unexpectedly passed validation")

error_locations = {tuple(item["loc"]) for item in validation_errors}
assert ("observed_at",) in error_locations
assert ("temperature_c",) in error_locations

There are schemas at many boundaries: instrument message, HTTP JSON, domain command, database table,
event/queue message, feature table, model input, and frontend state. They need not be identical.
Translate deliberately and version contracts independently when consumers change at different rates.

## 6. Repository: persistence behind a narrow interface

In [ ]:
repository = SQLiteMeasurementRepository(application_database_path)
stored_measurement, was_created = repository.save(valid_command)

assert was_created is True
assert stored_measurement.measurement_id == 1
assert stored_measurement.temperature_c == 84.5
assert repository.is_ready() is True

The repository owns parameterized SQL, transactions, row mapping, bounded ordering, and connection
lifecycle. The service does not know table column order. In a production PostgreSQL implementation,
the same behavioral contract needs integration tests against PostgreSQL; SQLite is not proof of another
engine's concurrency, isolation, types, migrations, or query plans.

### Idempotency makes a retry safe

In [ ]:
retried_measurement, retry_created = repository.save(valid_command)

assert retry_created is False
assert retried_measurement.measurement_id == stored_measurement.measurement_id
assert len(repository.latest(limit=100)) == 1

A client may time out after the server commits but before receiving the response. The client cannot
infer whether the write happened. A producer-owned idempotency key lets the server return the canonical
existing result on retry. “Retry POST” without this contract can duplicate side effects.

The key's scope, retention, payload-conflict behavior, and transactional uniqueness must be specified.

## 7. Service layer: domain policy without HTTP

In [ ]:
service = MeasurementService(repository)

assert service.classify_temperature(20.0) == "normal"
assert service.classify_temperature(80.0) == "elevated"
assert service.classify_temperature(100.0) == "critical"

latest_outputs = service.latest(limit=10)
assert latest_outputs[0].risk_level == "elevated"

The threshold rule is pure and easy to test. A real scientific or ML policy also requires provenance,
units, calibration assumptions, uncertainty, approved version, validation evidence, monitoring, and a
safe response to unavailable or out-of-distribution inputs. Clean software architecture does not make
an invalid threshold scientifically correct.

## 8. Backend: translate domain behavior into HTTP

HTTP is a contract, not a function call with punctuation:

- method communicates intent and retry/cache expectations;
- path identifies a versioned resource boundary;
- headers carry media type, authentication, tracing, caching, and negotiation metadata;
- status distinguishes created, successful replay, invalid input, unauthorized, forbidden, conflict,
  rate limited, unavailable, and unexpected failure;
- JSON has a schema and compatibility policy;
- timeout and cancellation are normal outcomes.

In [ ]:
api_payload = {
    "idempotency_key": "instrument-run-0044",
    "sensor_id": "sensor-beta",
    "observed_at": "2026-08-29T14:31:00Z",
    "temperature_c": 101.25,
}
created_response = client.post(
    "/api/v1/measurements",
    json=api_payload,
    headers={"x-request-id": "lecture-trace-0044"},
)

assert created_response.status_code == 201
assert created_response.headers["content-type"].startswith("application/json")
assert created_response.headers["x-request-id"] == "lecture-trace-0044"
assert created_response.json()["risk_level"] == "critical"

### The replay has a different HTTP outcome but the same identity

In [ ]:
replay_response = client.post("/api/v1/measurements", json=api_payload)

assert replay_response.status_code == 200
assert replay_response.json()["created"] is False
assert (
    replay_response.json()["measurement_id"]
    == created_response.json()["measurement_id"]
)
assert len(repository.latest(limit=100)) == 2

### API errors are part of the public schema

In [ ]:
invalid_response = client.post(
    "/api/v1/measurements",
    json=api_payload | {"temperature_c": -300.0},
)

assert invalid_response.status_code == 422
assert invalid_response.json()["detail"][0]["loc"][-1] == "temperature_c"
assert len(repository.latest(limit=100)) == 2

Do not expose stack traces, SQL, secret values, internal hostnames, or arbitrary exception text to a
client. Give users a stable error code/message and correlation ID; keep reviewed diagnostic detail in
access-controlled telemetry. Decide whether validation errors are safe to echo field-by-field.

## 9. OpenAPI makes part of the contract machine-readable

In [ ]:
openapi_response = client.get("/openapi.json")
openapi_document = openapi_response.json()
measurement_operation = openapi_document["paths"]["/api/v1/measurements"]["post"]
request_schema = measurement_operation["requestBody"]["content"][
    "application/json"
]["schema"]

assert openapi_response.status_code == 200
assert request_schema["$ref"].endswith("/MeasurementInput")
assert "201" in measurement_operation["responses"]
assert "MeasurementOutput" in openapi_document["components"]["schemas"]

OpenAPI supports generated clients, documentation, contract diffing, and consumer tests. It does not
capture every semantic invariant: idempotency scope, units, ordering, authorization, freshness, or
scientific meaning still need prose and executable examples. Treat incompatible schema changes as
product changes, not incidental refactors.

## 10. Liveness and readiness answer different questions

In [ ]:
live_response = client.get("/health/live")
ready_response = client.get("/health/ready")

assert live_response.json() == {"status": "alive"}
assert ready_response.json() == {"status": "ready"}
assert live_response.status_code == ready_response.status_code == 200

- **Liveness:** should the platform restart this process?
- **Readiness:** should the load balancer send new traffic here?
- **Startup:** has initialization completed?

A liveness check should not fail merely because a shared database briefly fails, or every replica may
restart together. A readiness check should be cheap, bounded, and honest about dependencies required
for the route. Health endpoints must not leak topology or credentials.

## 11. Correlation identifiers connect evidence across layers

In [ ]:
generated_id_response = client.get("/api/v1/measurements/latest")
safe_supplied_id_response = client.get(
    "/api/v1/measurements/latest",
    headers={"x-request-id": "phone-session-17"},
)
unsafe_supplied_id_response = client.get(
    "/api/v1/measurements/latest",
    headers={"x-request-id": "not safe because it has spaces"},
)

assert generated_id_response.headers["x-request-id"]
assert safe_supplied_id_response.headers["x-request-id"] == "phone-session-17"
assert (
    unsafe_supplied_id_response.headers["x-request-id"]
    != "not safe because it has spaces"
)

A request ID is not authentication and should not contain personal data. Validate or replace external
values before reflecting them in headers/logs. In a distributed system, propagate trace context and
attach the identifier to structured logs, metrics exemplars, queue messages, and downstream requests.

## 12. Frontend: another program and another failure boundary

In [ ]:
frontend_response = client.get("/")
frontend_source = frontend_response.text

assert frontend_response.status_code == 200
assert frontend_response.headers["content-type"].startswith("text/html")
assert "Scientific Measurement Monitor" in frontend_source
assert dashboard_html() == frontend_source
assert 'fetch("/api/v1/measurements/latest?limit=20"' in frontend_source

The browser first receives HTML, CSS, and JavaScript. JavaScript then makes a separate HTTP request for
JSON. These requests can be cached, fail, race, time out, or observe different deployed versions.

A useful interface models at least loading, empty, success, stale, partial, validation, authorization,
rate-limit, server-error, timeout, and offline states. A spinner forever is not an error strategy.

### Render untrusted data as text

In [ ]:
assert ".textContent =" in frontend_source
assert ".innerHTML" not in frontend_source
assert "if (!response.ok)" in frontend_source
assert "AbortController" in frontend_source
assert "finally" in frontend_source

The sample uses `textContent`, not `innerHTML`, for API values. This removes one common DOM injection
path but is not a complete frontend threat model. Apply contextual output encoding, a restrictive
Content Security Policy, dependency review, secure cookies, CSRF defenses where relevant, and careful
URL construction. Never place a privileged cloud/database secret in browser JavaScript.

## 13. Responsive, accessible behavior is part of correctness

In [ ]:
responsive_contracts = (
    'name="viewport"',
    "width=device-width",
    "@media (min-width: 48rem)",
    "grid-template-columns: 1fr",
    "min-height: 2.75rem",
    "prefers-reduced-motion",
    'aria-live="polite"',
)

assert all(contract in frontend_source for contract in responsive_contracts)

Mobile-first layout begins with a usable narrow normal flow and enhances when content needs space—not
for one fashionable phone model. The viewport tag lets mobile browsers use device-width CSS layout.
Touch targets, zoom, text reflow, keyboard navigation, focus, contrast, reduced motion, screen-reader
announcements, locale, timezone, and slow networks are functional requirements.

## 14. Phone and OS “emulation” has several meanings

| Technique | What it changes | What it cannot prove |
| --- | --- | --- |
| resize/DevTools responsive mode | viewport and quick layout feedback | real browser/OS/hardware behavior |
| Playwright browser context | viewport, user agent, touch, DPR, locale, timezone, permissions | every native device behavior |
| Android emulator | Android system image and virtual hardware | exact physical device/performance |
| iOS Simulator | simulated iOS environment on macOS | real radio, sensors, hardware, all WebKit differences |
| cloud device farm | remote real devices or emulators | every user environment; can be costly/flaky |
| physical device | actual hardware/browser/network/accessibility | broad coverage by itself |

Simulation, emulation, and real-device testing are complementary. State exactly which layer ran.

In [ ]:
device_contract_matrix = (
    {
        "name": "desktop-chromium",
        "viewport": (1440, 900),
        "touch": False,
        "locale": "en-US",
        "timezone": "America/Chicago",
    },
    {
        "name": "narrow-android-chromium",
        "viewport": (360, 800),
        "touch": True,
        "locale": "en-US",
        "timezone": "America/Chicago",
    },
    {
        "name": "narrow-ios-webkit",
        "viewport": (390, 844),
        "touch": True,
        "locale": "es-US",
        "timezone": "America/Los_Angeles",
    },
)

assert all(profile["viewport"][0] > 0 for profile in device_contract_matrix)
assert {profile["name"] for profile in device_contract_matrix} == {
    "desktop-chromium",
    "narrow-android-chromium",
    "narrow-ios-webkit",
}

### A user-agent header is not a phone emulator

In [ ]:
contract_results = {}
for profile in device_contract_matrix:
    response = client.get(
        "/api/v1/measurements/latest?limit=10",
        headers={"user-agent": f"course-emulation/{profile['name']}"},
    )
    contract_results[profile["name"]] = (response.status_code, response.json())

assert {status for status, _ in contract_results.values()} == {200}
serialized_bodies = {
    json.dumps(body, sort_keys=True) for _, body in contract_results.values()
}
assert len(serialized_bodies) == 1

This checks that the API contract does not accidentally branch on our artificial user agent. It does
not render pixels, run WebKit/Chromium, emulate touch, test a virtual keyboard, or reproduce an OS.
A real browser suite would use Playwright projects plus selected physical-device checks.

```typescript
// Illustrative Playwright configuration—not executed by this Python notebook.
projects: [
  { name: "desktop-chromium", use: { ...devices["Desktop Chrome"] } },
  { name: "android-chromium", use: { ...devices["Pixel 7"] } },
  { name: "ios-webkit", use: { ...devices["iPhone 15"] } },
]
```

Add tests for loading/error states, keyboard-only use, accessible names, orientation, locale, timezone,
dark mode, reduced motion, offline/slow network, and API-version compatibility. Avoid asserting every
pixel; combine semantic assertions, targeted screenshots, and human accessibility review.

## 15. Operating-system compatibility is a different matrix

In [ ]:
workflow_path = project_root / ".github" / "workflows" / "course-ci.yml"
workflow_source = workflow_path.read_text(encoding="utf-8")

for runner in ("ubuntu-latest", "macos-latest", "windows-latest"):
    assert runner in workflow_source

assert "uv sync --locked" in workflow_source
assert "uv run pytest -q" in workflow_source

An OS CI matrix runs Python/package tests on different hosted operating systems. It does not prove a
Safari mobile UI works, because browser engines and devices are another dimension. Cross-platform code
still uses `pathlib`, temporary directories, Unicode-safe text, explicit timezones, and no shell-specific
assumptions.

## 16. API evolution requires producer–consumer compatibility

In [ ]:
latest_response = client.get("/api/v1/measurements/latest?limit=10")
latest_payload = latest_response.json()
required_mobile_fields = {
    "measurement_id",
    "sensor_id",
    "observed_at",
    "temperature_c",
    "risk_level",
}

assert latest_payload
assert required_mobile_fields <= latest_payload[0].keys()
assert latest_response.status_code == 200

Mobile apps may remain installed for months. A backend cannot assume every client deploys with it.
Prefer additive response evolution, tolerant readers where safe, explicit deprecation windows, and
consumer contract tests. Breaking semantics may require `/api/v2`; adding a path version alone does not
create a compatibility strategy.

Database, event, API, model, and UI migrations need overlapping compatibility windows.

## 17. Authentication, authorization, CORS, and CSP differ

- **Authentication:** who/what is making the request?
- **Authorization:** may this identity perform this action on this resource?
- **CORS:** which browser origins may read a cross-origin response? It is not API authentication.
- **CSRF:** can a malicious origin induce an authenticated browser action?
- **CSP:** which frontend resources/scripts may execute or load?
- **Rate limiting/quotas:** how is abuse or accidental overload bounded?

The sample intentionally implements no login. Production authentication belongs to a reviewed identity
design, not a hard-coded teaching token. Prefer same-origin frontend/API hosting when appropriate; for
cross-origin deployments, allow exact trusted origins rather than reflexively using `*`.

## 18. Local topology and production topology are not identical

```text
local test
  TestClient → ASGI app → service → temporary SQLite

local development
  browser → localhost Uvicorn → service → local/dev database

production
  user/device
    → DNS → CDN/WAF/TLS/load balancer or API gateway
    → replicated application compute
    → managed database / object store / queue
    → logs, metrics, traces, alerts
```

Local fidelity should be sufficient for fast feedback, while integration/staging tests cover platform
differences. Attempting to perfectly reproduce the cloud on every laptop can create a second platform
that is expensive and still inaccurate.

### Common implementation options and when they fit

| Role | Common options | Selection questions |
| --- | --- | --- |
| Python backend | FastAPI, Django, Flask | typed APIs, batteries included, team familiarity, ecosystem |
| frontend | server-rendered HTML, lightweight JS, React, Vue, Svelte | interaction complexity, accessibility, team skill, bundle/ops cost |
| transactional data | PostgreSQL/MySQL, managed variants, document stores | transactions, query model, consistency, operations, portability |
| analytical data | warehouse, lake/lakehouse, query engine | scale, concurrency, governance, latency, scan/compute cost |
| asynchronous work | SQS/RabbitMQ/Kafka/Kinesis/Pub/Sub | queue vs replayable log, ordering, throughput, retention |
| compute | VM, PaaS, container service, serverless functions, Kubernetes | workload duration, scaling, portability, operational staff |
| static frontend | backend-served assets, object store + CDN, managed frontend host | same origin, cache, deployment independence, previews |
| telemetry | OpenTelemetry plus Prometheus/Grafana, cloud-native suite, commercial APM | standards, correlation, retention, privacy, cost |

There is no universally modern stack. Choose the fewest components that meet measured requirements and
that the team can operate safely.

### Deployment diagram: one common production shape

```mermaid
flowchart LR
    Client[Browser / phone / instrument] --> DNS[DNS]
    DNS --> Edge[CDN + WAF + TLS]
    Edge -->|static assets| Frontend[(Versioned frontend assets)]
    Edge -->|/api/*| Gateway[Load balancer / API gateway]
    Gateway --> App1[API instance A]
    Gateway --> App2[API instance B]
    App1 --> DB[(Managed relational database)]
    App2 --> DB
    App1 --> Queue[[Queue / stream]]
    App2 --> Queue
    App1 -. logs metrics traces .-> Observe[Telemetry platform]
    App2 -. logs metrics traces .-> Observe
    Client -. RUM errors web vitals .-> Observe
```

The database is private, application instances are replaceable, static assets are immutable, and both
server and client evidence carry release and trace identifiers.

## 19. Hosting means assigning operational responsibilities

An AWS-shaped mapping might use Route 53 for DNS, CloudFront/S3 for a static frontend, WAF and API
Gateway or an Application Load Balancer at the edge, Lambda/ECS/EKS/EC2 for compute, RDS/Aurora for
relational state, S3 for objects, SQS/Kinesis for asynchronous work, Secrets Manager/parameter services
for configuration, and CloudWatch/X-Ray/OpenTelemetry-compatible collection for telemetry.

Equivalent roles exist on Azure, Google Cloud, institutional Kubernetes/HPC, and managed platforms.
Choosing fewer managed components is often optimal for a small team. Vendor service count is not an
architecture quality metric.

### DNS, TLS, edge, compute, and storage fail independently

Ask who owns certificate renewal, process startup/restart, replica count, autoscaling, health probes,
database backup/restore, encryption keys, secret rotation, network policy, patching, quotas, cost alerts,
and on-call response. “Hosted in the cloud” answers none of these by itself.

## 20. Process, server, image, container, and host are different

- Uvicorn starts one or more **server processes** for an ASGI application.
- An OCI **image** is an immutable filesystem/config artifact.
- A **container** is a running isolated process created from an image; it shares a host kernel.
- A VM has a guest operating system; a host may run many containers.
- An orchestrator/platform schedules instances, restarts them, routes traffic, and injects config.

Containers improve artifact consistency; they do not automatically provide HTTPS, backups, zero
downtime, security, observability, or correct scaling.

In [ ]:
illustrative_containerfile = """FROM python:3.12-slim@sha256:<reviewed-digest>
WORKDIR /app
COPY pyproject.toml uv.lock ./
COPY src ./src
RUN <locked-production-install-command>
USER 10001
CMD ["uvicorn", "deployed_app:app", "--host", "0.0.0.0", "--port", "8080"]
"""

assert "@sha256:" in illustrative_containerfile
assert "USER 10001" in illustrative_containerfile
assert "--reload" not in illustrative_containerfile
assert "COPY . ." not in illustrative_containerfile

This is a review exercise, not a complete Dockerfile: replace placeholders, define `deployed_app`, add
a `.dockerignore`, build for the target architecture, scan dependencies/image, generate provenance/SBOM,
and test the exact image. Never bake credentials or production data into an image. Run as non-root with
a read-only filesystem and minimal permissions when the platform allows.

## 21. Configuration changes by environment; code and artifacts do not

In [ ]:
deployment_configuration = {
    "environment": "staging",
    "database_secret_reference": "secret://measurement-db/staging",
    "log_level": "INFO",
    "request_timeout_seconds": 5,
    "allowed_origins": ("https://staging.example.invalid",),
}

required_configuration = {
    "environment",
    "database_secret_reference",
    "log_level",
    "request_timeout_seconds",
    "allowed_origins",
}
assert required_configuration == deployment_configuration.keys()
assert "password" not in deployment_configuration
assert deployment_configuration["request_timeout_seconds"] > 0

Store non-secret configuration in reviewed deployment configuration and resolve secrets through an
approved identity/secret manager at runtime. Environment variables are a transport, not a secret
manager. Validate configuration at startup, redact diagnostics, rotate credentials, and scope each
environment/account separately.

## 22. Database migrations constrain release order

Use expand–migrate–contract for a backward-incompatible change:

1. **Expand:** add a compatible nullable column/table/index; old code still works.
2. Deploy code that writes/reads old and new forms as needed.
3. **Migrate/backfill:** bounded, observable, restartable, and verified.
4. Deploy code that no longer requires the old form.
5. **Contract:** remove old schema only after every consumer is compatible.

Rolling back application code does not automatically roll back data or a destructive migration. Test
restore procedures, not only backups.

## 23. CI builds evidence; CD promotes an artifact

```text
commit / pull request
  → format + lint + type/static checks
  → unit + property + contract tests
  → database/browser integration tests
  → build wheel + frontend assets + container image
  → vulnerability/license/provenance checks
  → publish immutable artifact once
  → deploy same digest to staging
  → migrations + smoke + selected E2E
  → approval/policy
  → progressive production rollout
  → health/SLO checks → continue or rollback/roll forward
```

Do not rebuild different source for production. Promotion should identify the exact artifact digest,
code revision, dependency lock, schema/model versions, configuration revision, approver, and results.

### CI/CD production flow and feedback loops

```mermaid
flowchart LR
    Commit[Commit / PR] --> CI[CI: lint, test, scan, build]
    CI --> Artifact[(Signed immutable artifact + provenance)]
    Artifact --> Staging[Deploy same digest to staging]
    Staging --> Migration[Compatible migrations]
    Migration --> E2E[Smoke + selected E2E + policy gate]
    E2E --> Canary[Canary production rollout]
    Canary --> Observe{SLOs, errors, traces, client RUM healthy?}
    Observe -->|yes| Promote[Increase traffic]
    Promote --> Observe
    Observe -->|no| Mitigate[Stop, rollback, or roll forward]
    Mitigate --> Incident[Incident review + regression test]
    Incident --> Commit
```

The deployment pipeline changes production; therefore concurrency, credentials, approvals, audit,
timeouts, cancellation, and partial-failure behavior are production code concerns.

In [ ]:
with (project_root / "pyproject.toml").open("rb") as handle:
    project_document = tomllib.load(handle)

lock_bytes = (project_root / "uv.lock").read_bytes()
release_manifest = {
    "distribution": project_document["project"]["name"],
    "version": project_document["project"]["version"],
    "lock_sha256": hashlib.sha256(lock_bytes).hexdigest(),
    "api_version": application.version,
    "database_schema_version": 1,
    "model_policy": "temperature-threshold-teaching-v1",
}

assert len(release_manifest["lock_sha256"]) == 64
assert release_manifest["api_version"] == "1.0.0"

### Continuous delivery is not continuous deployment

Continuous integration merges small changes with automated evidence. Continuous delivery keeps a
validated artifact releasable, often with approval. Continuous deployment automatically promotes
every qualifying change. None means “skip review,” and deployment success means neither user value nor
scientific validity.

Use least-privilege deployment identities, protected environments, concurrency control, signed or
attested artifacts, and a tested response to partial rollout.

## 24. Rollouts reduce blast radius

- **Rolling:** replace instances gradually; versions overlap.
- **Blue/green:** prepare a parallel environment and switch traffic.
- **Canary:** expose a small traffic slice first and compare health/SLOs.
- **Feature flag:** separate code deployment from feature exposure; flags add state and cleanup work.
- **Shadow:** copy traffic without returning candidate responses; protect privacy and side effects.

A rollback condition must be measurable before release. Database/event changes may require roll-forward
rather than binary rollback.

## 25. Observability connects user symptoms to system causes

- **Logs:** discrete structured events with timestamp, severity, request/trace ID, operation, safe
  dimensions, and error category.
- **Metrics:** aggregatable rates, errors, latency distributions, saturation, queue depth, freshness,
  data quality, and model behavior.
- **Traces:** causally connected spans across service/database/queue boundaries.

Avoid high-cardinality identifiers as metric labels and sensitive payloads in logs. Telemetry has its
own retention, access, cost, and failure behavior.

In [ ]:
def safe_request_event(
    *, request_id: str, route: str, status_code: int, duration_ms: float
) -> dict[str, str | int | float]:
    """Build a small structured event without request payloads or credentials."""
    return {
        "event": "http_request_completed",
        "request_id": request_id,
        "route": route,
        "status_code": status_code,
        "duration_ms": round(duration_ms, 3),
    }


request_event = safe_request_event(
    request_id="lecture-trace-0044",
    route="POST /api/v1/measurements",
    status_code=created_response.status_code,
    duration_ms=12.34567,
)

assert request_event["duration_ms"] == 12.346
assert "payload" not in request_event
assert "temperature_c" not in request_event

### Monitor clients, not only servers

A backend can report 99.99% success while users see a blank screen because an asset, JavaScript
exception, unsupported browser, stale cache, or client network fails before the API call. Client-side
evidence completes—but does not replace—server observability.

| Need | Typical tools/options | Evidence |
| --- | --- | --- |
| local diagnosis | browser DevTools Console, Network, Performance, Accessibility | request waterfall, console error, layout/performance profile |
| error tracking | Sentry, Rollbar, Bugsnag, commercial APM | exception, stack, release, route, safe context |
| real-user monitoring | CloudWatch RUM, Datadog/New Relic Browser, Grafana Faro, Elastic RUM | page/API latency, failure rate, Web Vitals, device/browser class |
| distributed tracing | OpenTelemetry browser/server instrumentation and compatible backend | trace/span IDs across browser, API, DB, queue |
| product analytics | approved privacy-aware event platform | user journey completion and feature behavior |
| synthetic monitoring | Playwright/browser probes from scheduled locations | critical journey availability before users report it |
| session replay | carefully governed replay product | visual interaction context, with high privacy risk |

Names are examples, not endorsements. Prefer open semantic conventions and an export path so operational
evidence is not trapped in one vendor.

### A client event needs a privacy and cardinality contract

Record release/build, route template, coarse device/browser class, operation, duration, outcome/error
code, trace/request ID, and timestamp. Sample successful high-volume events; retain errors more heavily.
Do not collect form contents, access tokens, full URLs with query data, raw scientific records, or stable
personal/device identifiers by default. Scrub before transport, not after storage.

Source maps help decode minified JavaScript but may expose source; upload them privately to the error
service and associate them with the exact frontend release. Monitor telemetry delivery failures without
making monitoring block the user's action.

In [ ]:
client_event = {
    "event": "latest_measurements_load",
    "frontend_release": "web-2026.08.29.1",
    "route": "/monitor",
    "device_class": "narrow-touch",
    "browser_family": "webkit",
    "duration_ms": 184.2,
    "outcome": "success",
    "request_id": "phone-session-17",
}
forbidden_client_fields = {
    "access_token",
    "authorization",
    "email",
    "raw_payload",
    "temperature_c",
}

assert forbidden_client_fields.isdisjoint(client_event)
assert client_event["frontend_release"]
assert client_event["duration_ms"] >= 0

### Correlation diagram: symptom to cause

```mermaid
sequenceDiagram
    participant B as Browser/RUM
    participant E as Edge
    participant A as API/OpenTelemetry
    participant D as Database
    participant O as Logs + metrics + traces
    B->>E: GET /api/v1/... trace context
    E->>A: forward request ID / trace context
    A->>D: traced bounded query
    D-->>A: result or timeout
    A-->>B: status + request ID
    B-->>O: sampled outcome, release, duration, safe device class
    A-->>O: structured event, RED metrics, trace span
    E-->>O: edge status and latency
```

An investigator begins with the user's release/time/request ID, follows the trace, checks aggregate
metrics for scope, and then reads relevant structured events. Logs alone are not a monitoring strategy.

### Service-level objectives turn reliability into a decision

In [ ]:
requests = 50_000
failed_requests = 37
availability = 1 - failed_requests / requests
slo_target = 0.999
error_budget = requests * (1 - slo_target)
error_budget_remaining = error_budget - failed_requests

assert availability == 0.99926
assert round(error_budget) == 50
assert round(error_budget_remaining) == 13

An SLI measures behavior; an SLO sets a target; an SLA is a business/legal commitment. Choose user-
meaningful indicators such as successful fresh responses within a latency threshold. Average latency
hides tails. Alert on actionable symptoms and budget burn, with a runbook and owner—not every isolated
exception.

## 26. Data and ML add another lifecycle

```text
versioned source snapshot
 → validated feature pipeline
 → train/evaluate candidate
 → registry: model + code + environment + data/feature lineage
 → approval
 → batch or online serving artifact
 → prediction logs, outcomes, drift/performance/safety monitoring
 → retraining proposal (not blind automatic replacement)
```

Offline and online feature definitions must agree in units, missingness, windows, leakage boundary, and
version. A frontend threshold label and a model prediction are different products unless the contract
says otherwise.

In [ ]:
feature_contract = {
    "name": "temperature_c",
    "dtype": "float64",
    "unit": "degree Celsius",
    "valid_range": (-273.15, 2_000.0),
    "freshness_seconds": 60,
    "null_policy": "reject",
    "version": 1,
}

assert feature_contract["unit"] == "degree Celsius"
assert feature_contract["valid_range"][0] == -273.15
assert feature_contract["freshness_seconds"] > 0

### Batch, online, and streaming are operational choices

Batch scoring maximizes throughput and reproducibility but may be stale. Online scoring serves one or
small groups under a latency budget and needs bounded dependencies. Streaming processes unbounded
events with ordering, watermark, lateness, duplicate, checkpoint, and replay semantics.

Do not call an unbounded event stream “real time” without a latency/freshness definition. Do not put a
slow training job in a synchronous user request.

## 27. Queues decouple work but introduce delivery semantics

In [ ]:
retry_policy = {
    "maximum_attempts": 5,
    "base_delay_seconds": 1.0,
    "maximum_delay_seconds": 30.0,
    "jitter": "full",
    "retryable": ("timeout", "connection-reset", "temporary-unavailable"),
    "terminal": ("invalid-schema", "unauthorized", "forbidden"),
    "dead_letter_after_exhaustion": True,
}

assert retry_policy["maximum_attempts"] > 1
assert set(retry_policy["retryable"]).isdisjoint(retry_policy["terminal"])
assert retry_policy["dead_letter_after_exhaustion"] is True

At-least-once delivery requires idempotent consumers or deduplication. Acknowledging before durable
work risks loss; acknowledging after work can redeliver after a crash. Bound retries with exponential
backoff and jitter, route poison messages for investigation, monitor queue age/depth, and apply
backpressure rather than accepting unlimited work.

## 28. Threat-model the complete path

Consider spoofed instrument identity, replay, schema bombs, oversized bodies, SQL/DOM injection,
broken object authorization, cross-site requests, leaked tokens, dependency compromise, malicious
model files, sensitive telemetry, public storage, denial of service, and overprivileged deployment.

Use TLS, strong workload/user identity, object-level authorization, input/body limits, parameterized
queries, safe rendering, rate limits, network segmentation, secret rotation, dependency/artifact
verification, audit, retention, backups, and incident exercises. Security controls need tests at the
layer where they actually operate.

## 29. Performance is an end-to-end budget

In [ ]:
latency_budget_ms = {
    "edge_and_tls": 40,
    "application": 25,
    "database": 55,
    "serialization": 10,
    "frontend_render": 70,
}
total_budget_ms = sum(latency_budget_ms.values())

assert total_budget_ms == 200
assert all(duration > 0 for duration in latency_budget_ms.values())

Measure tail latency at the user boundary and spans inside. Avoid N+1 queries, unbounded responses,
excessive JSON, blocking work in the request path, and retries that multiply load. Cache only with an
explicit key, freshness, invalidation, authorization, and stampede policy. Autoscaling cannot repair a
query whose cost grows without bound.

## 30. Test each risk at the narrowest credible boundary

| Test | Real boundary | Typical purpose |
| --- | --- | --- |
| unit/property | function/class | domain cases and invariants |
| contract | producer/consumer schema | compatibility without full deployment |
| repository integration | application + real DB engine | SQL, migrations, transactions |
| component/vertical slice | API + service + test DB | request-to-storage behavior |
| browser component | DOM + mocked/controlled API | UI states and accessibility |
| deployed E2E | browser/device → hosted system | critical journey and wiring |
| smoke | newly deployed environment | minimal readiness after rollout |
| synthetic | scheduled production-like request | continuing external availability |

Test doubles improve control but reduce realism. Real dependencies improve fidelity but add state,
latency, cost, and failure. Make the tradeoff visible in the test name and documentation.

### Execute one honest vertical-slice test

In [ ]:
journey_payload = {
    "idempotency_key": "instrument-run-journey-0001",
    "sensor_id": "sensor-gamma",
    "observed_at": "2026-08-29T14:35:00Z",
    "temperature_c": 72.0,
}

journey_create = client.post("/api/v1/measurements", json=journey_payload)
journey_latest = client.get("/api/v1/measurements/latest?limit=100")
journey_frontend = client.get("/")
persisted_ids = {
    item.idempotency_key for item in repository.latest(limit=100)
}

assert journey_create.status_code == 201
assert "instrument-run-journey-0001" in persisted_ids
assert any(
    row["idempotency_key"] == "instrument-run-journey-0001"
    for row in journey_latest.json()
)
assert journey_frontend.status_code == 200

This crosses HTTP routing, Pydantic, service, repository, SQL, and frontend asset loading. It does not
execute the frontend JavaScript or cross a socket, DNS, TLS, container, managed database, IAM, CDN, or
real browser. Therefore we call it a vertical-slice component test, not “the E2E test.”

## 31. Debugging: follow evidence in request order

1. Reproduce with environment, artifact version, time, account, client, and request ID.
2. Identify the first incorrect observable boundary: UI state, HTTP, API model, policy, SQL, or data.
3. Inspect the matching trace and structured events without exposing secrets.
4. Compare schema/model/migration/config versions across instances.
5. Check dependency saturation: pool, queue, CPU, memory, disk, quota, network, downstream health.
6. Reduce to the narrowest failing contract and add a regression test.
7. Mitigate user harm before optimizing the perfect root-cause explanation.

Do not debug production by editing a running container or database manually without an approved,
audited incident procedure.

### Common failure modes

| Symptom | Likely boundary | First evidence |
| --- | --- | --- |
| browser says network error | DNS/TLS/CORS/offline/API | browser network panel + request ID |
| 422 response | producer/API schema | safe validation details + client version |
| duplicate row | retry/idempotency transaction | key uniqueness + request history |
| old client breaks | API compatibility | OpenAPI diff + client version matrix |
| readiness flaps | dependency/pool/timeout | health latency + saturation |
| desktop works, phone fails | viewport/touch/WebKit/network | browser/device matrix |
| staging works, production fails | config/IAM/network/data scale | manifest/config/trace comparison |
| deployment green, users fail | insufficient test boundary | external SLI and synthetic journey |
| predictions shift | data/feature/model contract | lineage, drift, outcomes, model version |

## Guided practice: draw the complete journey

For the measurement monitor, draw the write and read paths. At every arrow label:

1. payload/schema and owner;
2. timeout, retry, idempotency, and ordering;
3. authentication/authorization;
4. version compatibility;
5. log/metric/trace evidence;
6. narrow test and broader test; and
7. user-visible response to failure.

**Success criterion:** every “end-to-end” claim names its endpoints and real/doubled dependencies.

In [ ]:
critical_boundaries = {
    "producer_to_api": {"schema", "identity", "idempotency", "timeout"},
    "api_to_service": {"validated_types", "authorization", "domain_error"},
    "service_to_database": {"transaction", "uniqueness", "query_bound"},
    "api_to_browser": {"status", "json_schema", "compatibility", "cache"},
}

assert all(len(contracts) >= 3 for contracts in critical_boundaries.values())
assert "idempotency" in critical_boundaries["producer_to_api"]

## Guided practice: design the phone/browser matrix

Choose a risk-based minimum matrix across Chromium/WebKit/Firefox, narrow/wide viewports, touch and
keyboard, locale/timezone, reduced motion, dark mode, slow/offline network, and supported client
versions. Label each row DevTools simulation, Playwright emulation, OS simulator/emulator, cloud device,
or physical device.

**Success criterion:** no row claims more realism than it provides, and critical accessibility/error
states have assertions.

## Independent practice: productionize one deployment

Select one target—AWS managed services, another cloud, institutional Kubernetes, or a small managed
platform—and write an architecture decision record covering:

- expected users, latency, throughput, availability, data scale, and budget;
- DNS/TLS/edge, frontend hosting, API compute, database, objects, queues, and telemetry;
- network and IAM boundaries, secrets, encryption, backup/restore, and retention;
- build artifact, migration, rollout, smoke test, rollback/roll-forward, and on-call ownership;
- desktop/mobile/browser/OS coverage; and
- model/data lineage, monitoring, and scientific validation.

**Success criterion:** every managed service has an owned responsibility and measurable failure plan.

## Independent practice: add a compatible API field

Add `unit: "degree Celsius"` to the response without breaking an older consumer:

1. change the public model and mapping;
2. update OpenAPI/contract tests;
3. keep old required fields stable;
4. update frontend rendering safely;
5. test desktop and mobile browser projects; and
6. describe deployment order and rollback.

**Success criterion:** repository rows remain private, old clients still work, and the unit cannot drift
between backend and frontend.

## Extension: incident game day

Simulate one failure at a time: database unavailable, exhausted pool, schema mismatch, slow query,
expired certificate, broken DNS, queue backlog, stale feature, unavailable model, or frontend/API
version mismatch. Define expected user behavior, alert, trace, mitigation, recovery, and regression test.

**Success criterion:** the exercise tests people, runbooks, permissions, communication, and restoration—not
only exception handling.

## Extension: infrastructure as code

Represent hosting resources, IAM, network rules, alarms, dashboards, and environment configuration as
reviewed, tested code. Plan changes before applying them; pin providers/modules; separate environments;
scan policy; detect drift; protect state; and require approval for destructive changes.

Infrastructure code can reproduce a bad architecture perfectly. Pair syntax tests with policy,
integration, restore, capacity, and cost checks.

## 32. Release disposable resources

In [ ]:
client.close()
application_workspace.cleanup()

assert not application_database_path.exists()

## Retrieval practice

Answer without executing code:

1. Distinguish data, request, training, serving, and deployment pipelines.
2. Why must an end-to-end claim name its endpoints and real dependencies?
3. Trace one measurement from JSON to database and back to visible UI state.
4. Why is a timeout after POST ambiguous, and how does idempotency help?
5. Distinguish liveness, readiness, and startup checks.
6. What does OpenAPI capture, and which semantics remain outside it?
7. Why is CORS not authentication?
8. Distinguish a server process, image, container, VM, and hosting platform.
9. Why can database migration make application rollback unsafe?
10. Distinguish rolling, blue/green, canary, feature-flag, and shadow rollout.
11. Distinguish logs, metrics, and traces.
12. Why is a changed user-agent header not phone emulation?
13. What can Playwright emulate, and what still needs a simulator or device?
14. Why is an OS CI matrix different from a browser/device matrix?
15. Which evidence identifies the exact running code, data, schema, and model?

## Takeaway

```text
scientific meaning
  → versioned schema and validated command
  → idempotent service and transactional repository
  → explicit HTTP contract
  → safe, responsive, accessible client states
  → immutable artifact and compatible migration
  → progressive hosting rollout
  → logs + metrics + traces + user-centered SLO
  → reproducible data/model lineage and accountable operation
```

End to end is not a synonym for “large” or “realistic.” It is a precise claim about the path and
boundaries exercised. Strong teams combine small comprehensible components, executable contracts,
carefully chosen broad tests, and honest operational evidence.

## Further reading

- [Python `sqlite3`](https://docs.python.org/3/library/sqlite3.html)
- [FastAPI testing](https://fastapi.tiangolo.com/tutorial/testing/)
- [FastAPI deployment concepts](https://fastapi.tiangolo.com/deployment/concepts/)
- [FastAPI containers](https://fastapi.tiangolo.com/deployment/docker/)
- [Pydantic models](https://docs.pydantic.dev/latest/concepts/models/)
- [MDN responsive design](https://developer.mozilla.org/en-US/docs/Learn_web_development/Core/CSS_layout/Responsive_Design)
- [MDN viewport metadata](https://developer.mozilla.org/en-US/docs/Web/HTML/Reference/Elements/meta/name/viewport)
- [MDN accessibility](https://developer.mozilla.org/en-US/docs/Web/Accessibility)
- [Playwright emulation](https://playwright.dev/docs/emulation)
- [Docker: what is a container?](https://docs.docker.com/get-started/docker-concepts/the-basics/what-is-a-container/)
- [Docker building images](https://docs.docker.com/get-started/docker-concepts/building-images/)
- [OpenTelemetry documentation](https://opentelemetry.io/docs/)
- [OWASP API Security Top 10](https://owasp.org/API-Security/)
- [The Twelve-Factor App](https://12factor.net/)